## 第6章　矩阵即变换 —— 从"表格"到"空间扭曲"

> 本章目标：理解矩阵乘法的两种视角——代数（行×列的点积组合）和几何（空间扭曲：基向量被映射到新位置）。手撕全连接层的前向传播 `output = input @ weight.T + bias`，并掌握广播机制——PyTorch 中最容易"静默出错"的规则。
> 前置知识：第 5 章（点积）、第 4 章（向量运算）、第 3 章（张量 shape 与轴）



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
print("env ready")


### 6.1　矩阵×向量 —— 一次"空间变换"

你已经知道矩阵是一堆向量叠成的表格（第 3.3 节），也知道点积衡量两个向量的相似度（第 5 章）。**矩阵×向量就是把这两者结合起来：用矩阵的每一行分别与向量做点积，结果拼成一个新向量。**

$$**W** **x** = \begin{bmatrix} w_{11} & w_{12} \\ w_{21} & w_{22} \\ w_{31} & w_{32} \end{bmatrix} \begin{bmatrix} x_1 \\ x_2 \end{bmatrix} = \begin{bmatrix} **w**_1·**x** \\ **w**_2·**x** \\ **w**_3·**x** \end{bmatrix}$$

看这个公式：输入是一个 2 维向量，输出是一个 3 维向量——**矩阵乘法的本质是把一个空间的向量"映射"到另一个空间。** 维度从 2 变成了 3。这就是"线性变换"这个名字的来源：变换前在 ℝ² 空间中，变换后在 ℝ³ 空间中。

📐 **定义　矩阵-向量乘法（Matrix-Vector Multiplication）**：**W** (m×n) × **x** (n,) → **y** (m,)。**y**[i] = **W**[i, :]·**x**（矩阵第 i 行与 **x** 的点积）。前提：矩阵的列数 = 向量的维度（都是 n）。

💻 **代码　矩阵×向量的两种等价实现**


In [ ]:
import numpy as np

# 3×2 矩阵 × 2 维向量 → 3 维向量
W = np.array([[1.0, 2.0],
              [3.0, 4.0],
              [5.0, 6.0]])  # shape (3, 2)
x = np.array([2.0, 1.0])    # shape (2,)

# 方式 1: @ 运算符——PyTorch 标配
y_at = W @ x   # shape (3,)

# 方式 2: 手工逐行点积——理解底层发生了什么
y_manual = np.array([np.dot(W[i], x) for i in range(3)])

print(f"W (3×2) @ x (2,) → y (3,):")
print(f"  y = {y_at}")
print(f"  逐行点积: {y_manual}")
print(f"  两者一致: {np.allclose(y_at, y_manual)} ✓")
print(f"  含义: 第0行[1,2]·[2,1]=1×2+2×1=4")
print(f"        第1行[3,4]·[2,1]=3×2+4×1=10")
print(f"        第2行[5,6]·[2,1]=5×2+6×1=16")


> **关键洞察**：矩阵×向量就是 **n 次点积的批量执行**——矩阵的每一行和输入向量做一次点积，得到一个输出分量。全连接神经网络的一层就是 `output = W @ input + b`——W 的每一行对应一个输出神经元的"权重向量"。

🔗 **AI 连接**：`nn.Linear(784, 256)` 内部存储的就是一个 256×784 的权重矩阵 W。forward 时执行 `input @ W.T + bias`——每个输出神经元（256 个）与输入向量（784 维）做点积。第 12 章的反向传播会揭示"这个点积的梯度如何传回 W 的每个元素"。

---


### 6.2　矩阵×矩阵 —— 点积规模的又一次升级

矩阵×向量是一次空间变换。那矩阵×矩阵呢？**把多个向量打包，一次性全做变换。** C = A @ B 的含义是：A 的每一行和 B 的每一列做一次点积，结果填入 C 的对应位置。

$$C[i,j] = **A**\text{的第}i\text{行}·**B**\text{的第}j\text{列} = \sum_k A[i,k] \cdot B[k,j]$$

这是深度学习中 99% 计算量的来源——每一层前向传播都是一次矩阵乘法，每一个 attention 计算都是一次矩阵乘法。而 NumPy/PyTorch 的 `@` / `matmul` 用高度优化的 BLAS 库（Intel MKL / NVIDIA cuBLAS）在底层执行，比 Python 三重循环快 100-10000 倍。

📐 **定义　矩阵乘法（Matrix Multiplication）**：A (m×k) @ B (k×n) → C (m×n)。前提：**A 的列数 = B 的行数**（都等于 k）。C 的每个元素 = A 的某行与 B 的某列的点积。

💻 **代码　三重循环 vs @ 运算符的速度对比——见证向量化的威力**


In [ ]:
import numpy as np
import time

# 小矩阵便于理解，大矩阵便于感受速度差
m, k, n = 200, 300, 250

np.random.seed(42)
A = np.random.randn(m, k)
B = np.random.randn(k, n)

# ===== 方式 1: Python 三重循环——理解语义，慢到怀疑人生 =====
t0 = time.perf_counter()
C_loop = np.zeros((m, n))
for i in range(m):
    for j in range(n):
        # C[i,j] = A的第i行 与 B的第j列 的点积
        C_loop[i, j] = sum(A[i, kk] * B[kk, j] for kk in range(k))
t1 = time.perf_counter()

# ===== 方式 2: @ 运算符——一行，C 层 BLAS 执行 =====
t2 = time.perf_counter()
C_at = A @ B    # shape (200, 250)
t3 = time.perf_counter()

print(f"三重循环: {t1 - t0:.4f} 秒")
print(f"A @ B:    {t3 - t2:.4f} 秒")
print(f"加速比:   {(t1 - t0) / (t3 - t2):.0f} 倍")
print(f"结果一致: {np.allclose(C_loop, C_at)} ✓")
print(f"\nC 的 shape: {C_at.shape} ← (m={m}, n={n})，k={k} 被'消掉'了")
print(f"规则: (m×k) @ (k×n) → (m×n)，中间维度 k 必须匹配且会消失")


> **关键洞察**：三重循环是教给你"矩阵乘法在做什么"，`@` 运算符是教给你"工业界怎么做矩阵乘法"。两者的结果完全一致，但速度差 100-10000 倍——因为 NumPy 的 `@` 在底层调用了 BLAS（Basic Linear Algebra Subprograms），这是用 C/Fortran + 手工优化的 SIMD 汇编写的，经历了几十年的打磨。GPU 上的 cuBLAS 进一步把这种加速推到极致——这就是为什么 AI 训练需要 GPU：**本质上就是把 `A @ B` 这类操作做得极快。**

🔗 **AI 连接**：第 29 章 Transformer 中的 Q @ Kᵀ 就是 `(batch×heads, seq, d_k) @ (batch×heads, d_k, seq)`——批量矩阵乘法。PyTorch 的 `torch.matmul` 自动处理前面额外的 batch 和 heads 维度，底层依然走 cuBLAS。

---


### 6.3　几何视角 —— 矩阵把空间"扭成"新形状

代数的视角：矩阵乘法 = 行×列的点积组合。几何的视角：**矩阵乘法 = 把整个空间的网格扭曲成新的形状。** 原始空间中整齐的方格，经过矩阵变换后变成了倾斜、拉伸、甚至翻转的网格——而矩阵的每一列，恰好就是"原始坐标系的基向量被映射到了哪里"。

用大白话说：2×2 矩阵 `[[a, b], [c, d]]` 把 x 轴上的单位向量 [1, 0] 映射到 [a, c]，把 y 轴上的单位向量 [0, 1] 映射到 [b, d]。**整个空间的扭曲方式，完全由这两个新基向量的位置决定。**

📐 **定义　线性变换（Linear Transformation）的几何含义**：矩阵的每一列 = 原始标准基向量变换后的新位置。网格上所有点都按同样的规则被"带着走"。

💻 **代码　亲眼见证矩阵如何扭曲整个空间的网格**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ===== 三种经典变换矩阵 =====
matrices = {
    "旋转 45°": np.array([[0.707, -0.707],
                          [0.707,  0.707]]),
    "X轴拉伸2倍": np.array([[2.0, 0.0],
                            [0.0, 1.0]]),
    "剪切 (Shear)": np.array([[1.0, 0.5],
                              [0.0, 1.0]]),
}

# 原始网格点（10×10 的点阵）
grid_points = np.stack(np.meshgrid(np.linspace(-2, 2, 10),
                                   np.linspace(-2, 2, 10)), axis=-1)
# grid_points shape: (10, 10, 2) —— 每个位置一个 (x, y) 坐标对

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# --- 左 0: 原始网格 ---
for i in range(10):
    axes[0].plot(grid_points[i, :, 0], grid_points[i, :, 1],
                 'b-', alpha=0.4, linewidth=0.8)
    axes[0].plot(grid_points[:, i, 0], grid_points[:, i, 1],
                 'b-', alpha=0.4, linewidth=0.8)
axes[0].set_title("原始网格"); axes[0].set_aspect('equal')
axes[0].set_xlim(-4, 4); axes[0].set_ylim(-4, 4); axes[0].grid(alpha=0.2)

# --- 右 1-3: 变换后的网格 ---
for idx, (name, M) in enumerate(matrices.items(), 1):
    # 对网格上每个点施加变换: new_point = M @ point
    # grid_points (10,10,2) → 展平 → @ M.T → 还原形状
    flat = grid_points.reshape(-1, 2)       # (100, 2)
    transformed = (M @ flat.T).T            # (100, 2) — 每个点经过 M 变换
    transformed_grid = transformed.reshape(10, 10, 2)

    for i in range(10):
        axes[idx].plot(transformed_grid[i, :, 0],
                       transformed_grid[i, :, 1],
                       'r-', alpha=0.5, linewidth=0.8)
        axes[idx].plot(transformed_grid[:, i, 0],
                       transformed_grid[:, i, 1],
                       'r-', alpha=0.5, linewidth=0.8)
    # 标注基向量的去向
    e1_new = M @ np.array([1.0, 0.0])  # 原 x 轴去哪了
    e2_new = M @ np.array([0.0, 1.0])  # 原 y 轴去哪了
    axes[idx].quiver(0, 0, e1_new[0], e1_new[1], angles='xy',
                     scale_units='xy', scale=1, color='blue', width=0.04)
    axes[idx].quiver(0, 0, e2_new[0], e2_new[1], angles='xy',
                     scale_units='xy', scale=1, color='green', width=0.04)
    axes[idx].set_title(name); axes[idx].set_aspect('equal')
    axes[idx].set_xlim(-4, 4); axes[idx].set_ylim(-4, 4); axes[idx].grid(alpha=0.2)

plt.tight_layout(); plt.show()
print("蓝色箭头 = 原 x 轴[1,0]被映射到哪里")
print("绿色箭头 = 原 y 轴[0,1]被映射到哪里")
print("整个网格随基向量一起变形——这就是矩阵=空间扭曲")


> **关键洞察**：多层神经网络就是**多次空间扭曲的叠加**。第 1 层把原始数据空间扭一下（ReLU 激活把负半轴"折"到零），第 2 层在扭曲后的空间中再扭一下，层层叠加——最终把一团纠缠的原始数据点（猫的像素）扭成漂亮的可分状态（线性可分的类别）。第 8 章的特征值会揭示"扭曲有多大"，第 9 章的 SVD 会告诉你"怎么只保留最重要的扭曲方向"。

🔗 **AI 连接**：第 28 章的两层神经网络本质就是 `ReLU(X @ W1 + b1) @ W2 + b2`——两次空间变换 + 一次非线性折叠。你现在建立的"网格被矩阵扭来扭去"的直觉，会在第 28 章被翻译成"决策边界从直线弯成曲线"。

---


### 6.4　全连接层手撕 —— output = input @ weight.T + bias

现在把前三节的知识组装成一个真正的 AI 组件：全连接层（Fully Connected / Linear Layer）。PyTorch 中 `nn.Linear(d_in, d_out)` 做的事情只有一行：


In [ ]:
output = input @ weight.T + bias


其中 `weight.shape = (d_out, d_in)`（注意 PyTorch 存的是 `(out_features, in_features)`），`bias.shape = (d_out,)`。`input` 的 shape 是 `(batch, d_in)`。

你可能会困惑：为什么是 `@ weight.T` 而不是 `@ weight`？因为 PyTorch 的 `nn.Linear` 内部存储的 weight 是 `(d_out, d_in)`，但矩阵乘法规则要求 `(batch, d_in) @ (d_in, d_out)`。所以需要转置：`(batch, d_in) @ (d_in, d_out)` → `(batch, d_out)`。这和本章 6.1 节"矩阵每一行和 x 做点积"的逻辑完全一致——只是现在"每一行"变成了 weight 转置后的列。

📐 **定义　全连接层（Fully Connected Layer）**：`output = input @ Wᵀ + b`。W 的每一行是一个输出神经元的权重向量（与输入做点积），b 为每个输出神经元加一个偏移。

💻 **代码　手撕 nn.Linear —— 用 NumPy 实现，与 PyTorch 对比**


In [ ]:
import numpy as np

# 尝试导入 PyTorch 做对比（没有也无妨，NumPy 实现是独立的）
try:
    import torch
    import torch.nn as nn
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

np.random.seed(42)
batch_size, d_in, d_out = 32, 784, 256

# 模拟输入
X = np.random.randn(batch_size, d_in)

# ===== 手撕全连接层 =====
# PyTorch 的 nn.Linear(784, 256) 内部 weight shape = (256, 784)
W = np.random.randn(d_out, d_in) * np.sqrt(2.0 / d_in)  # Kaiming 初始化（第 25 章）
b = np.zeros(d_out)

# 前向传播——一行！
output_np = X @ W.T + b  # (32, 784) @ (784, 256) + (256,) → (32, 256)
print(f"NumPy 全连接层:")
print(f"  输入: {X.shape}")
print(f"  W.T: {W.T.shape} ← W 存储为 ({d_out},{d_in})，使用时转置")
print(f"  输出: {output_np.shape}")
print(f"  公式: ({batch_size},{d_in}) @ ({d_in},{d_out}) + ({d_out},) → ({batch_size},{d_out})")

# ===== 与 PyTorch 对比 =====
if HAS_TORCH:
    torch.manual_seed(42)
    layer = nn.Linear(d_in, d_out, bias=True)
    # 覆盖 PyTorch 的权重为我们手写的权重（保证可比）
    with torch.no_grad():
        layer.weight.copy_(torch.tensor(W, dtype=torch.float32))
        layer.bias.copy_(torch.tensor(b, dtype=torch.float32))
    X_torch = torch.tensor(X, dtype=torch.float32)
    output_torch = layer(X_torch)
    print(f"\nPyTorch nn.Linear 输出前 3 个值: {output_torch[0, :3].detach().numpy().round(4)}")
    print(f"NumPy 手撕输出前 3 个值:        {output_np[0, :3].round(4)}")
    print(f"两者一致: {np.allclose(output_np, output_torch.detach().numpy(), atol=1e-5)} ✓")
else:
    print("\n(PyTorch 未安装，NumPy 实现本身已完整可用)")


> **关键洞察**：全连接层的本质是：**把 784 维的输入"重新编码"为 256 维的隐藏表示。** 256 个神经元各自用自己的权重向量（784 维）与输入做点积——每个神经元检测输入中是否存在某个"模式"（权重向量指向的方向），若点积大（cosθ 接近 1），说明该模式强，该神经元激活值高。

🔗 **AI 连接**：第 28 章的两层网络 = 两次全连接层中间夹一个 ReLU 激活。第 29 章 Transformer 中的 Feed-Forward Network 也是全连接层：`d_model → 4×d_model → d_model`。你此刻手撕的这行代码，出现在了本书后面每一章的神经网络中。

---


### 6.5　广播机制（Broadcasting） —— PyTorch 的"自动形状对齐"

你在 6.4 节可能已经注意到一个"不对劲"的地方：


In [ ]:
output = X @ W.T + b   # (32, 256) + (256,)   ← 形状不匹配！


`(32, 256)` 是一个 32 行 256 列的矩阵，`(256,)` 是一个 256 维向量——它们的 shape 明明不同，NumPy/PyTorch 怎么就能相加？答案是**广播（Broadcasting）**：自动把 `(256,)` 扩展为 `(1, 256)` 再复制 32 行变成 `(32, 256)`，然后逐元素相加。这个"隐形的自动对齐"是 PyTorch 代码中最容易"静默出错"的规则——shape 对了但语义错了，代码不报错，结果悄悄变成垃圾。

📐 **定义　广播规则（Broadcasting Rules）**：两个张量进行逐元素运算时，从最后一维开始往前对齐——维度相等或其中之一为 1 则兼容，否则报错。缺失的维度自动在前面补 1。

口诀：**从右往左对齐，大小相等或为 1，缺失的维度补 1 再复制。**

💻 **代码　广播规则全景演示——什么能用，什么会炸**


In [ ]:
import numpy as np

# ===== 正确案例：AI 代码中最常见的广播 =====
# (batch, d_out) + (d_out,)  —— 偏置加在每个样本上
X_out = np.random.randn(32, 256)   # 32个样本 × 256维输出
b = np.random.randn(256)           # 1个偏置向量
result = X_out + b                 # (256,) → (1,256) → 广播到 (32,256)
print(f"案例1: ({X_out.shape}) + ({b.shape}) → {result.shape}")
print(f"  含义: 256维偏置被复制32份，每个样本都加上同一个偏置 ✓")

# ===== 正确案例：不同形状——但从右对齐后兼容 =====
a = np.ones((3, 1))    # (3, 1)
b = np.ones((1, 4))    # (1, 4)
c = a + b              # (3, 1) + (1, 4) → 各自广播 → (3, 4)
print(f"\n案例2: ({a.shape}) + ({b.shape}) → {c.shape}")
print(f"  对齐: (3,1) vs (1,4) → 都广播为 (3,4) ✓")

# ===== 正确案例：标量广播 =====
d = np.ones((3, 4))
e = d + 10.0           # 标量 10 → 广播到 (3,4) 的每个元素
print(f"\n案例3: ({d.shape}) + 标量 → {e.shape} ✓")

# ===== 错误案例：不兼容的 shape =====
try:
    f = np.ones((3, 4))
    g = np.ones((3, 2))
    h = f + g   # (3,4) + (3,2) → 从右对齐: 4 vs 2，不等也不为1 → 爆炸！
except ValueError as e:
    print(f"\n案例4 (错误): (3,4) + (3,2) → {str(e)[:60]}...")

# ===== 实战：Transformer 中的广播 =====
# (batch, seq_len, d_model) + (d_model,) —— LayerNorm 的 bias
batch, seq_len, d_model = 32, 10, 512
X_transformer = np.random.randn(batch, seq_len, d_model)
ln_bias = np.random.randn(d_model)
result_ln = X_transformer + ln_bias  # (d_model,) → 广播到 (batch,seq_len,d_model)
print(f"\nTransformer 广播: ({X_transformer.shape}) + ({ln_bias.shape}) → {result_ln.shape} ✓")
print(f"  含义: 每个 token 的 512 维嵌入都加上同一个偏置向量")

# ===== 口诀总结 =====
print(f"\n📐 广播口诀:")
print(f"  1. 从右往左对齐 shape")
print(f"  2. 每个维度: 相等 || 有一个是 1 → 兼容")
print(f"  3. 缺失的维度自动补 1")
print(f"  4. 兼容则各自广播到统一 shape 再运算")


> **关键洞察**：广播机制是 AI 代码中最容易"静默出错"的规则——shape 对了但语义错了不会报错。例如你想给一个 `(batch, features)` 矩阵的每一列加不同的数，如果你传的向量 shape 是 `(batch,)` 而非 `(features,)`，广播会在错误的方向上对齐，结果全部算错但不会抛异常。**写 AI 代码时，每次看到不同 shape 的张量做加减乘除，第一反应就应该是"广播规则怎么对齐的？"**

🔗 **AI 连接**：第 29 章 Transformer 的 attention mask 广播——`(batch, 1, seq, seq)` 的 mask 广播到 `(batch, heads, seq, seq)` 的 scores；第 21 章 LayerNorm 的 `(d_model,)` 参数广播到 `(batch, seq, d_model)` 的每个 token。每次广播都在你此刻建立的"从右往左对齐"规则下严格执行。

---


### 6.6　Agent 服务化：连续批处理 —— 矩阵乘法的工程威力

现在把矩阵乘法的视角拉到一个实际的 Agent 部署场景：**你有一个 LLM 服务，同时收到 100 个用户的请求——每个用户的消息长度不同。你怎么高效地批量处理？**

逐个串行处理？100 个用户排队，第 100 个用户等到崩溃。全部 padding 到最大长度？如果 99 个用户只发了 10 个 token，1 个用户发了 2000 个 token，padding 会让 99% 的计算浪费在无效的 `<pad>` token 上。

**连续批处理（Continuous Batching）** 是工业界的答案。它的核心思想只有一句话：**把所有用户的有效 token 拼接成一个"锯齿形"的大批量矩阵，用一次矩阵乘法同时处理——然后用 Attention Mask 隔离不同用户的 token，确保用户 A 的 token 不会注意到用户 B 的 token。**

这本质就是你刚学的矩阵乘法 + 广播机制 + 掩码操作的三合一工程应用。`(total_valid_tokens, d_model)` 的大矩阵一次性投入 GPU，矩阵乘法并行处理所有 token——不管它们来自哪个用户。

📐 **定义　连续批处理（Continuous Batching）**：将多个变长序列的有效 token 拼接为一个批量矩阵 `(total_tokens, d_model)`，通过构造块对角 Attention Mask 隔离不同序列。计算量 ∝ 总有效 token 数，而非 batch×max_len。相比 padding 方案可节省 2-10 倍计算。

💻 **代码　模拟连续批处理：拼接 + Mask + 批量矩阵乘法**


In [ ]:
import numpy as np

np.random.seed(42)

# ===== 模拟 4 个用户，各自的消息长度不同 =====
seq_lens = [3, 5, 2, 4]           # 用户0:3词, 用户1:5词, 用户2:2词, 用户3:4词
d_model = 8
total_tokens = sum(seq_lens)       # 14 个有效 token（不是 4×5=20!）

# 生成每个用户的 token 嵌入
user_embeddings = []
for length in seq_lens:
    user_embeddings.append(np.random.randn(length, d_model))

# ===== 拼接为连续批量矩阵 =====
X_batched = np.vstack(user_embeddings)  # (14, 8)
print(f"连续批处理: {total_tokens} 个有效 token")
print(f"Padding 方案: {4 * max(seq_lens)} 个位置 ({4*max(seq_lens) - total_tokens} 个被浪费)\n")
print(f"拼接矩阵 shape: {X_batched.shape}  ← (total_tokens, d_model)")

# ===== 构造 Attention Mask（块对角结构） =====
# 每个用户只能看到自己序列内的 token，不能跨用户注意
mask = np.zeros((total_tokens, total_tokens))
offset = 0
for length in seq_lens:
    mask[offset:offset+length, offset:offset+length] = 1  # 块内可见
    offset += length

print(f"\nAttention Mask (1=可见, 0=屏蔽):")
print(mask.astype(int))
print(f"\n解读: 用户0(token 0-2)只能看到自己; 用户1(token 3-7)只能看到自己; ...")
print(f"      块对角结构确保'用户隔离'——这就是连续批处理的核心。")

# ===== 模拟一次批量 Q·K 计算 =====
W_Q = np.random.randn(d_model, d_model) * 0.1
W_K = np.random.randn(d_model, d_model) * 0.1
Q = X_batched @ W_Q  # (14, 8) @ (8, 8) = (14, 8) —— 一次矩阵乘法处理所有用户!
K = X_batched @ W_K

scores = Q @ K.T / np.sqrt(d_model)  # (14, 14) —— 所有 token 对之间的得分
# 应用 mask：不允许跨用户注意 → 将 mask=0 的位置设为 -inf
scores_masked = np.where(mask == 1, scores, -np.inf)

print(f"\nscores[0, 4] (用户0 token0 → 用户1 token1): {scores[0, 4]:.4f}")
print(f"scores_masked[0, 4] (被 mask 屏蔽后): {scores_masked[0, 4]:.4f} ← -inf，无法注意!")
print(f"scores_masked[0, 1] (用户0内部): {scores_masked[0, 1]:.4f} ← 正常")


> **关键洞察**：连续批处理的数学本质，就是你刚学的矩阵乘法的工程威力展示——`(total_tokens, d) @ (d, d) = (total_tokens, d)`。14 个来自不同用户的 token（而不是 20 个位置）被投入同一个矩阵乘法，GPU 的并行计算单元全部被有效利用。**这才是第 6 章矩阵乘法在工业 Agent 服务中的最终落地——不是教科书上的数学，而是每天支撑百万并发请求的工程基座。**

🔗 **AI 连接**：vLLM、TensorRT-LLM、SGLang 等推理框架都实现了连续批处理。第 29 章 Transformer 的 Attention Mask 原理（Causal Mask + 块对角用户隔离）直接建立在本节的 mask 构造之上。第 5.5 节 RAG 检索的结果 + 本节的连续批处理 + 第 4.5 节的状态迁移 = Agent 服务的完整推理栈。

---


**✏️ 习题**

> 在下方新建代码单元格作答。
1. （概念）矩阵乘法 `A @ B` 的前提条件是什么？结果矩阵 C 的第 i 行第 j 列元素 C[i,j] 是怎么算出来的？
2. （概念）为什么 `(3, 4) @ (4, 5)` 可以算，但 `(3, 4) @ (3, 4)` 会报错？用矩阵乘法的维度规则解释。
3. （概念）广播规则的三条核心法则是什么？`(32, 1, 10) + (1, 5, 10)` 的结果 shape 是什么？为什么？
4. （代码）对 3×3 矩阵 A（旋转矩阵）和向量 x，计算 `A @ x`，验证变换后向量的长度不变（旋转不改变长度）。
5. （概念）连续批处理中，为什么 Attention Mask 必须是"块对角"结构？如果把 mask 全部设为 1（允许跨用户注意）会有什么后果？用一句话回答。
6. （代码）用三重循环、`np.matmul`、`np.einsum` 三种方式计算同一个 `(100, 200) @ (200, 150)` 矩阵乘法，对比耗时，验证结果一致。
7. （代码）构造一个广播场景：一组成绩数据 shape=(50, 4)（50 个学生，4 科成绩），每科有不同的满分和权重。用广播实现成绩标准化和加权求和，打印最终得分最高的 3 个学生的索引和分数。
---
> 🔗 **章末钩子**：矩阵乘法让你能"批量做点积"。但有些矩阵可以"逆向操作"——给定变换后的结果，能倒推出原始输入吗？这就是逆矩阵要解决的问题。
> 预览：下一章——**矩阵的逆与线性方程组**，解方程的艺术。
